# CBOW - Embedding - Word2Vec

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim

In [30]:
# Define model
class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embed_size):
        super(CBOWModel, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_size)
        self.linear = nn.Linear(embed_size, vocab_size)

    def forward(self, context):
        context_embeds = self.embeddings(context).sum(dim=0)
        output = self.linear(context_embeds)
        return output

In [31]:
context_size = 2
raw_text = "word embeddings are really awesome and powerful"
tokens = raw_text.split()
vocab = set(tokens)
word_to_index = {word: i for i, word in enumerate(vocab)}
data = []
for i in range(2, len(tokens) - 2):
    context = [word_to_index[word] for word in tokens[i - 2:i] + tokens[i + 1:i + 3]]
    target = word_to_index[tokens[i]]
    data.append((torch.tensor(context), torch.tensor(target)))

In [32]:
# Hyperparameters
vocab_size = len(vocab)
embed_size = 10
learning_rate = 0.01
epochs = 100

# Initialize CBOW model
cbow_model = CBOWModel(vocab_size, embed_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(cbow_model.parameters(), lr=learning_rate)

In [33]:
# Training Loop
for epoch in range(epochs):
    total_loss = 0
    for context, target in data:
        optimizer.zero_grad()
        output = cbow_model(context)
        loss = criterion(output.unsqueeze(0), target.unsqueeze(0))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}, Loss: {total_loss}")

Epoch 1, Loss: 7.498630881309509
Epoch 2, Loss: 6.40700101852417
Epoch 3, Loss: 5.4730857610702515
Epoch 4, Loss: 4.6674270033836365
Epoch 5, Loss: 3.967672288417816
Epoch 6, Loss: 3.361325353384018
Epoch 7, Loss: 2.8429309129714966
Epoch 8, Loss: 2.4091123044490814
Epoch 9, Loss: 2.0542699098587036
Epoch 10, Loss: 1.7690033912658691
Epoch 11, Loss: 1.5414085388183594
Epoch 12, Loss: 1.3594494462013245
Epoch 13, Loss: 1.2126613855361938
Epoch 14, Loss: 1.0927385985851288
Epoch 15, Loss: 0.9933982789516449
Epoch 16, Loss: 0.9099826514720917
Epoch 17, Loss: 0.8390524089336395
Epoch 18, Loss: 0.778047651052475
Epoch 19, Loss: 0.7250466644763947
Epoch 20, Loss: 0.6785831898450851
Epoch 21, Loss: 0.6375252604484558
Epoch 22, Loss: 0.6009867489337921
Epoch 23, Loss: 0.5682635009288788
Epoch 24, Loss: 0.5387912839651108
Epoch 25, Loss: 0.5121110677719116
Epoch 26, Loss: 0.48784738779067993
Epoch 27, Loss: 0.46568791568279266
Epoch 28, Loss: 0.44537293165922165
Epoch 29, Loss: 0.42668332159519

In [34]:
# example usage
word_to_lookup = "embeddings"
word_index = word_to_index[word_to_lookup]
embedding = cbow_model.embeddings(torch.tensor([word_index]))
print(f"Embedding for '{word_to_lookup}': {embedding.detach().numpy()}")

Embedding for 'embeddings': [[ 1.4548568   0.07622258 -1.16858     0.9726893   1.0951145  -0.49633664
   0.6034088  -1.6360781   0.1934628  -0.03660405]]


In [35]:
print("Number of training samples:", len(data))

Number of training samples: 3


In [46]:
from gensim.models import Word2Vec

sentences = [
    ["word", "embeddings", "are", "really", "awesome"],
    ["embeddings", "capture", "semantic", "meaning"]
]

model = Word2Vec(
    sentences,
    vector_size=100,
    window=2,
    min_count=1,
    sg=0  # CBOW
)

vector = model.wv["embeddings"]
print(vector)

[-5.3622725e-04  2.3643136e-04  5.1033497e-03  9.0092728e-03
 -9.3029495e-03 -7.1168090e-03  6.4588725e-03  8.9729885e-03
 -5.0154282e-03 -3.7633716e-03  7.3805046e-03 -1.5334714e-03
 -4.5366134e-03  6.5540518e-03 -4.8601604e-03 -1.8160177e-03
  2.8765798e-03  9.9187379e-04 -8.2852151e-03 -9.4488179e-03
  7.3117660e-03  5.0702621e-03  6.7576934e-03  7.6286553e-04
  6.3508903e-03 -3.4053659e-03 -9.4640139e-04  5.7685734e-03
 -7.5216377e-03 -3.9361035e-03 -7.5115822e-03 -9.3004224e-04
  9.5381187e-03 -7.3191668e-03 -2.3337686e-03 -1.9377411e-03
  8.0774371e-03 -5.9308959e-03  4.5162440e-05 -4.7537340e-03
 -9.6035507e-03  5.0072931e-03 -8.7595852e-03 -4.3918253e-03
 -3.5099984e-05 -2.9618145e-04 -7.6612402e-03  9.6147433e-03
  4.9820580e-03  9.2331432e-03 -8.1579173e-03  4.4957981e-03
 -4.1370760e-03  8.2453608e-04  8.4986202e-03 -4.4621765e-03
  4.5175003e-03 -6.7869602e-03 -3.5484887e-03  9.3985079e-03
 -1.5776526e-03  3.2137157e-04 -4.1406299e-03 -7.6826881e-03
 -1.5080082e-03  2.46979

# skip-gram

In [45]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')  

sample = "Word embeddings are dense vector representations of words."
tokenized_corpus = word_tokenize(sample.lower()) 

skipgram_model = Word2Vec(sentences=[tokenized_corpus],
                          vector_size=100,  
                          window=5,         
                          sg=1,             
                          min_count=1,      
                          workers=4)        

# Training
skipgram_model.train([tokenized_corpus], total_examples=1, epochs=10)
skipgram_model.save("skipgram_model.model")
loaded_model = Word2Vec.load("skipgram_model.model")
vector_representation = loaded_model.wv['word']
print("Vector representation of 'word':", vector_representation)

Vector representation of 'word': [-9.5800208e-03  8.9437785e-03  4.1664648e-03  9.2367809e-03
  6.6457358e-03  2.9233587e-03  9.8055992e-03 -4.4231843e-03
 -6.8048164e-03  4.2256550e-03  3.7299085e-03 -5.6668529e-03
  9.7035142e-03 -3.5551414e-03  9.5499391e-03  8.3657773e-04
 -6.3355025e-03 -1.9741615e-03 -7.3781307e-03 -2.9811086e-03
  1.0425397e-03  9.4814906e-03  9.3598543e-03 -6.5986011e-03
  3.4773252e-03  2.2767992e-03 -2.4910474e-03 -9.2290826e-03
  1.0267317e-03 -8.1645092e-03  6.3240929e-03 -5.8001447e-03
  5.5353874e-03  9.8330071e-03 -1.5987856e-04  4.5296676e-03
 -1.8086446e-03  7.3613892e-03  3.9419360e-03 -9.0095028e-03
 -2.3953868e-03  3.6261671e-03 -1.0080514e-04 -1.2024897e-03
 -1.0558038e-03 -1.6681013e-03  6.0541567e-04  4.1633579e-03
 -4.2531900e-03 -3.8336846e-03 -5.0755290e-05  2.6549282e-04
 -1.7014991e-04 -4.7843382e-03  4.3120929e-03 -2.1710952e-03
  2.1056964e-03  6.6702347e-04  5.9686624e-03 -6.8418151e-03
 -6.8183104e-03 -4.4762432e-03  9.4359247e-03 -1.593

[nltk_data] Downloading package punkt to /home/asish-jose/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
